# 02 — Feature Engineering

This notebook demonstrates the full feature extraction pipeline for vibration signals:

1. **Signal cleaning** — bandpass filtering, wavelet denoising, normalisation
2. **Time-domain features** — RMS, kurtosis, crest factor, Shannon entropy, and 11 more
3. **Frequency-domain features** — spectral centroid, entropy, band energy ratios, dominant frequency
4. **Feature importance** — correlation with RUL, PCA visualisation

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from scipy import signal as scipy_signal
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

%matplotlib inline
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

np.random.seed(42)

## 1. Signal Cleaning

In [ ]:
from preprocessing.signal_cleaning import (
    normalize_zscore, normalize_minmax,
    butterworth_bandpass, butterworth_lowpass,
    remove_dc_offset
)

# Synthetic vibration signal: fundamental + harmonics + noise
fs = 10000.0  # 10 kHz
t = np.linspace(0, 0.1, int(fs * 0.1), endpoint=False)
fault_freq = 150.0  # Hz — simulated bearing fault frequency

clean_signal = (
    np.sin(2 * np.pi * fault_freq * t)         # fundamental
    + 0.4 * np.sin(2 * np.pi * 2 * fault_freq * t)  # 2nd harmonic
    + 0.2 * np.sin(2 * np.pi * 3 * fault_freq * t)  # 3rd harmonic
).astype(np.float32)

# Add DC offset + broadband noise
noisy_signal = clean_signal + 0.5 + 0.3 * np.random.randn(len(t)).astype(np.float32)

# Apply processing steps
dc_removed = remove_dc_offset(noisy_signal)
bandpass_filtered = butterworth_bandpass(dc_removed, lowcut=50, highcut=3000, fs=fs, order=4)
normalised = normalize_zscore(bandpass_filtered)

# Plot
fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
n_show = 1000

axes[0].plot(t[:n_show], noisy_signal[:n_show], color='gray', linewidth=0.6, label='Raw (noisy + DC)')
axes[0].set_title('Step 1 — Raw Signal')
axes[0].legend()

axes[1].plot(t[:n_show], dc_removed[:n_show], color='steelblue', linewidth=0.6, label='DC removed')
axes[1].set_title('Step 2 — DC Offset Removed')
axes[1].legend()

axes[2].plot(t[:n_show], bandpass_filtered[:n_show], color='coral', linewidth=0.6, label='Bandpass (50–3000 Hz)')
axes[2].set_title('Step 3 — Butterworth Bandpass Filter')
axes[2].legend()

axes[3].plot(t[:n_show], normalised[:n_show], color='forestgreen', linewidth=0.6, label='Z-score normalised')
axes[3].set_title('Step 4 — Z-Score Normalisation')
axes[3].set_xlabel('Time (s)')
axes[3].legend()

plt.tight_layout()
plt.show()

## 2. Time-Domain Feature Extraction

In [ ]:
from preprocessing.vibration_features import extract_time_features, TimeFeatures

# Compare healthy vs faulty signal features
healthy = np.random.randn(1024).astype(np.float32) * 0.3        # low amplitude
faulty  = (healthy + 0.5 * np.random.choice([-1, 1], 1024) *   # impulses
           np.random.exponential(1, 1024)).astype(np.float32)

feats_healthy = extract_time_features(healthy)
feats_faulty  = extract_time_features(faulty)

print('Feature comparison — Healthy vs Faulty signal:')
print(f"{'Feature':<25} {'Healthy':>10} {'Faulty':>10} {'Ratio':>8}")
print('-' * 56)
h_dict = feats_healthy.to_dict()
f_dict = feats_faulty.to_dict()
for k in h_dict:
    h, f = h_dict[k], f_dict[k]
    ratio = f / (h + 1e-10)
    print(f"{k:<25} {h:>10.4f} {f:>10.4f} {ratio:>8.2f}x")

In [ ]:
# Feature evolution across simulated degradation stages
from preprocessing.vibration_features import extract_time_features_batch

n_stages = 100
n_samples = 1024
features_over_time = []

for i in range(n_stages):
    degradation = i / n_stages
    # Signal amplitude increases and becomes more impulsive with degradation
    sig = (0.3 + degradation * 0.7) * np.random.randn(n_samples)
    if degradation > 0.5:
        n_impulses = int(degradation * 20)
        impulse_idx = np.random.choice(n_samples, n_impulses, replace=False)
        sig[impulse_idx] += np.random.choice([-1, 1], n_impulses) * (1 + degradation * 3)
    features_over_time.append(extract_time_features(sig.astype(np.float32)).to_array())

features_over_time = np.stack(features_over_time)
feature_names = TimeFeatures.feature_names()

# Plot key features over degradation
key_features = ['rms', 'kurtosis', 'crest_factor', 'entropy']
key_indices = [feature_names.index(f) for f in key_features]

fig, axes = plt.subplots(2, 2, figsize=(12, 7))
axes = axes.ravel()
stages = np.arange(n_stages)

for idx, (feat_name, feat_idx) in enumerate(zip(key_features, key_indices)):
    axes[idx].plot(stages, features_over_time[:, feat_idx], color='steelblue')
    axes[idx].set_title(feat_name.replace('_', ' ').title())
    axes[idx].set_xlabel('Degradation stage')
    axes[idx].axvline(50, color='red', linestyle='--', alpha=0.5, label='Onset')
    axes[idx].legend()

fig.suptitle('Time-Domain Feature Evolution During Degradation', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Frequency-Domain Feature Extraction

In [ ]:
from preprocessing.spectral_features import (
    extract_spectral_features, compute_fft, compute_envelope_spectrum
)

# Compute FFT of healthy vs faulty signal
freqs_h, mag_h = compute_fft(healthy, fs=10000.0)
freqs_f, mag_f = compute_fft(faulty,  fs=10000.0)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].semilogy(freqs_h[:500], mag_h[:500], color='steelblue', linewidth=0.8)
axes[0].set_xlabel('Frequency (Hz)')
axes[0].set_ylabel('Magnitude')
axes[0].set_title('FFT — Healthy Signal')

axes[1].semilogy(freqs_f[:500], mag_f[:500], color='coral', linewidth=0.8)
axes[1].set_xlabel('Frequency (Hz)')
axes[1].set_title('FFT — Faulty Signal (more broadband energy)')

plt.tight_layout()
plt.show()

In [ ]:
# Spectral feature comparison
sp_healthy = extract_spectral_features(healthy, fs=10000.0)
sp_faulty  = extract_spectral_features(faulty,  fs=10000.0)

print('Spectral Feature comparison — Healthy vs Faulty:')
print(f"{'Feature':<25} {'Healthy':>12} {'Faulty':>12}")
print('-' * 52)
for k in sp_healthy.to_dict():
    h = sp_healthy.to_dict()[k]
    f = sp_faulty.to_dict()[k]
    print(f"{k:<25} {h:>12.4f} {f:>12.4f}")

## 4. Feature Importance and PCA

In [ ]:
# Build a feature matrix across degradation stages
from preprocessing.spectral_features import extract_spectral_features_batch

# Use the time-domain features computed earlier
X_features = features_over_time                     # (100, 15)
y_rul = np.linspace(125, 0, n_stages)               # simulated RUL

# Pearson correlation of each feature with RUL
corr_with_rul = np.array([
    np.corrcoef(X_features[:, i], y_rul)[0, 1]
    for i in range(X_features.shape[1])
])

fig, ax = plt.subplots(figsize=(12, 4))
x_pos = np.arange(len(feature_names))
colors_bar = ['steelblue' if c >= 0 else 'coral' for c in corr_with_rul]
ax.bar(x_pos, corr_with_rul, color=colors_bar, alpha=0.85, edgecolor='white')
ax.axhline(0, color='black', linewidth=0.8)
ax.set_xticks(x_pos)
ax.set_xticklabels(feature_names, rotation=45, ha='right')
ax.set_ylabel('Pearson correlation with RUL')
ax.set_title('Feature–RUL Correlation (Time-Domain Features)')
plt.tight_layout()
plt.show()

In [ ]:
# PCA: project 15 features to 2D, colour by degradation stage
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_features)

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

fig, ax = plt.subplots(figsize=(8, 6))
sc = ax.scatter(X_pca[:, 0], X_pca[:, 1], c=y_rul, cmap='RdYlGn', s=30, alpha=0.9)
plt.colorbar(sc, ax=ax, label='RUL')
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)')
ax.set_title('PCA of Time-Domain Features — Coloured by RUL')
plt.tight_layout()
plt.show()

print(f'Cumulative variance explained by 2 PCs: {sum(pca.explained_variance_ratio_)*100:.1f}%')

## Summary

- **Time-domain features** most correlated with degradation: RMS, kurtosis, crest factor (captures impulsiveness)
- **Spectral features** capture the frequency content shift as fault harmonics develop
- **PCA** shows that the first 2 principal components already explain substantial variance, with clear RUL ordering

**Next:** [03 — LSTM RUL Prediction](03_lstm_rul_prediction.ipynb)